In [1]:
import time
from labjack import ljm

# ==========================================
# HARDWARE & CALIBRATION CONSTANTS
# ==========================================
# LabJack Registers
STEAM_REG = "AIN7"
TUBE_REG = "AIN6"

# LJTick-CurrentShunt Scaling Constant
LJTCS_SLOPE = 8.475

# Expected Voltage Limits from LJTCS for 4-20mA
V_MIN = 0.472  # 4mA baseline
V_MAX = 2.360  # 20mA baseline

# Low-End Noise Cutoff (in mA)
ZERO_CUTOFF_MA = 4.05

# Sensor Engineering Units (PSI Limits)
STEAM_MIN_PSI = 0.0
STEAM_MAX_PSI = 150.0

TUBE_MIN_PSI = 0.0
TUBE_MAX_PSI = 15.0


def scale_pressure(voltage, p_min, p_max):
    """Converts raw voltage to mA and scales to PSI with a zero-floor cutoff."""
    # 1. Convert voltage to mA using LJTCS transfer equation
    current_mA = LJTCS_SLOPE * voltage

    # 2. Enforce zero floor condition
    if current_mA <= ZERO_CUTOFF_MA:
        psi = 0.0
    else:
        # 3. Linear interpolation mapping: Volts to PSI
        psi = p_min + (voltage - V_MIN) * (p_max - p_min) / (V_MAX - V_MIN)

        # Safety clamp to prevent minor over-range fluctuations past max capacity
        if psi > p_max:
            psi = p_max

    return current_mA, psi


def main():
    try:
        # Open connection to the T7
        handle = ljm.openS("T7", "ANY", "ANY")
        info = ljm.getHandleInfo(handle)
        print(f"Connected to LabJack T7 [Serial: {info[2]}]")
        print("Monitoring Pressure Systems...")
        print("-" * 80)
        
        # Header layout for two distinct data blocks
        print(
            f"{'--- STEAM PRESSURE (AIN7) ---':<38} | {'--- TUBE DELTA P (AIN6) ---':<38}"
        )
        print(
            f"{'Volt(V)':<8}{'Current(mA)':<12}{'Pressure(PSI)':<15} | "
            f"{'Volt(V)':<8}{'Current(mA)':<12}{'Pressure(PSI)':<15}"
        )
        print("-" * 80)

        # Configure Analog Input Channels (Single-Ended, +/-10V Scale)
        for reg in [STEAM_REG, TUBE_REG]:
            ljm.eWriteName(handle, f"{reg}_NEGATIVE_CH", 199)
            ljm.eWriteName(handle, f"{reg}_RANGE", 10.0)

        while True:
            # --- Read and Process Steam Pressure (AIN7) ---
            v_steam = ljm.eReadName(handle, STEAM_REG)
            mA_steam, psi_steam = scale_pressure(v_steam, STEAM_MIN_PSI, STEAM_MAX_PSI)

            # --- Read and Process Tube Delta P (AIN6) ---
            v_tube = ljm.eReadName(handle, TUBE_REG)
            mA_tube, psi_tube = scale_pressure(v_tube, TUBE_MIN_PSI, TUBE_MAX_PSI)

            # --- Print Synchronized Data Matrix ---
            # Using carriage return '\r' to smoothly update data in place
            print(
                f"{v_steam:<8.3f}{mA_steam:<12.2f}{psi_steam:<15.2f} | "
                f"{v_tube:<8.3f}{mA_tube:<12.2f}{psi_tube:<15.2f}",
                end="\r",
            )

            time.sleep(0.5)

    except ljm.LJMError as e:
        print(f"\nLabJack LJM Error: {e}")
    except KeyboardInterrupt:
        print("\nTesting stopped by user.")
    finally:
        # Cleanly disconnect from hardware
        if "handle" in locals():
            ljm.close(handle)
            print("LabJack connection closed safely.       ")


if __name__ == "__main__":
    main()

Connected to LabJack T7 [Serial: 470042305]
Monitoring Pressure Systems...
--------------------------------------------------------------------------------
--- STEAM PRESSURE (AIN7) ---          | --- TUBE DELTA P (AIN6) ---           
Volt(V) Current(mA) Pressure(PSI)   | Volt(V) Current(mA) Pressure(PSI)  
--------------------------------------------------------------------------------
0.487   4.13        1.18            | 0.326   2.76        0.00           
Testing stopped by user.
LabJack connection closed safely.       
